In [ ]:
#Import libraries
import pandas as pd
from datetime import date
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# 2.1 DealerSKU query
query_dealersku = """
SELECT 
    RUN_DATE,
    RUN_CYCLE_ID,
    DEALER,
    SKU,
    LAST_N_DAY_SALES,
    SAFETY_STOCK,
    PREDICTED_SALES_SKU,
    LEAD_TIME_STOCK,
    CURRENT_AVAILABLE_STOCK,
    REORDER_LEVEL,
    MOP_PLANNED_QUANTITY,
    QUANTITY_DISPATCHED_TODAY,
    TOTAL_DISPATCH,
    TOTAL_B2G,
    TOTAL_OPEN_ORDER_QTY,
    MODIFIED_REORDER_URGENCY_SCORE,
    "Prorated_Quantity",
    "Prorated_B2G",
    PRIORITY_RANK_OVERALL,
    OBL_CR_LIMIT_AVAIL,
    BLOCK_STATUS_HHHD,
    BLOCK_STATUS_HHHG,
    BLOCK_STATUS_HHHU,
    BLOCK_STATUS_HM4N,
    BLOCK_STATUS_HM5V,
    BLOCK_STATUS_HM6C,
    CASE 
        WHEN OBL_CR_LIMIT_AVAIL >= 100000 THEN TRUE 
        ELSE FALSE 
    END AS FUND_STATUS
FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDealerSKUTable t
WHERE (t.run_date, t.run_cycle_id) IN (
    SELECT run_date, MAX(run_cycle_id)
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
    WHERE run_date = (
        SELECT MAX(run_date)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances
    )
    GROUP BY run_date
)
;
"""

# 2.2 DispatchableInstances query (fixed)
query_dispatchable = """
SELECT 
    RUN_DATE,
    RUN_CYCLE_ID,
    DEALER,
    SKU,
    PLANT,
    QUANTITY,
    MODEL,
    VERSION,
    PREDICTED_SALES_SKU,
    AVG_LEAD_TIME_STOCK      AS LEAD_TIME_STOCK,
    SAFETY_STOCK,
    REORDER_LEVEL,
    CURRENT_AVAILABLE_STOCK,
    CURRENT_MONTH_SALES,
    LAST_N_DAY_SALES,
    QUANTITY_DISPATCHED_TODAY,
    QUANTITY_DISPATCHED,
    TOTAL_OPEN_ORDER_QTY,
    B2G,
    LOAD_SIZE_MULTIPLES,
    AVAILABLE_PLANT_STOCK,
    TODAY_PRODUCTION_STOCK,
    PLANT_SKU_OPEN_ORDER_QTY,
    EOD_AVAILABLE_QUANTITY,
    NORMALISED_B2G_URGENCY_SCORE,
    FINAL_B2G               AS MIN_B2G_FILLABALE,
    "Prorated_Quantity",
    MAX_QTY_V1,
    "Normalized_MAX_QTY",
    MAX_QTY                 AS MAX_B2G_FILLABALE,
    PRIORITY_RANK_OVERALL,
    N,
    PRORITY_RANK_PLANT_WISE
FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DailyDispatchableInstances t
WHERE (t.run_date, t.run_cycle_id) IN (
    SELECT run_date, MAX(run_cycle_id)
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    WHERE run_date = (
        SELECT MAX(run_date)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    )
    GROUP BY run_date
)
;
"""

# 2.3 AllTrucks query
query_alltrucks = """
SELECT 
    RUN_DATE,
    RUN_CYCLE_ID,
    KEY_IDENTIFIER,
    DEALER,
    SKU,
    PLANT,
    MULTIPLIER,
    "Truck_no",
    UNITS_TO_BE_DISPATCHED,
    IS_COMPLETE_TRUCK,
    FUND_AVAILABLE,
    CUMMULATIVE_COST,
    B2G_URGENCY_SCORE,
    "Max_Score",
    "Median_Score",
    "Max_Score_Rank",
    "Median_Score_Rank"
FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.DAILYALLTRUCKS t
WHERE (t.run_date, t.run_cycle_id) IN (
    SELECT run_date, MAX(run_cycle_id)
    FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    WHERE run_date = (
        SELECT MAX(run_date)
        FROM MOP_DATABASE.DISPATCH_AUTOMATION_OBL.Daily_Unconstrained_file
    )
    GROUP BY run_date
)
;
"""

#Step 3 - Read the tables as pandas
df_dealersku = session.sql(query_dealersku).to_pandas()

df_dispatchable = session.sql(query_dispatchable).to_pandas()

df_alltrucks = session.sql(query_alltrucks).to_pandas()


# Step 4: Write all three DataFrames into a single Excel workbook
current_date = date.today().strftime("%Y-%m-%d")
output_filename = f"final_files_{current_date}.xlsx"

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    df_dealersku.to_excel(writer, sheet_name='DealerSKU', index=False)
    df_dispatchable.to_excel(writer, sheet_name='DispatchableInstances', index=False)
    df_alltrucks.to_excel(writer, sheet_name='AllTrucks', index=False)

import os
file_path = os.path.join(os.getcwd(), f"final_files_{current_date}.xlsx")
# session.file.put(file_path, "@DAILY_FILES_TRUCK_DISPATCH", auto_compress=False, overwrite=True)
# print(f"✅ Data successfully exported to {output_filename}")


In [ ]:
session.sql(f"PUT 'file://{file_path}' @DAILY_FILES_TRUCK_DISPATCH AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()